# Model Quality Estimation

## Project Overview

In the previous projects, the training and test samples were already prepared. In this project, the focus is different. Here, I want to understand how to design the evaluation process itself, not just train a model and report a score.

The main idea behind this project is that a model can look good on paper while still being evaluated in the wrong way. That can happen because of poor data splitting, leakage, weak validation logic, or unfair model selection. So in this notebook, I will focus on building a more reliable evaluation pipeline.

## Problem Statement

The main problem in this project is not only training a model, but making sure that the reported model quality is realistic and trustworthy.

A model can produce optimistic results when:
- the data is split incorrectly,
- future information leaks into training,
- repeated groups appear in both training and test sets,
- feature selection is done unfairly,
- or hyperparameters are tuned using information that should not be available.

Because of that, this project is really about learning how to evaluate models correctly and how to choose the right validation strategy for the data.

## Project Objective

In this notebook, I want to build a clear and reliable evaluation workflow for this machine learning task.

More specifically, I will:
- review the theoretical concepts behind validation, feature selection, and hyperparameter optimization,
- preprocess the dataset and create the required features,
- implement custom split methods,
- implement custom cross-validation methods,
- compare my implementations with sklearn,
- apply multiple feature selection methods,
- tune model hyperparameters,
- and finally compare all results to choose the best overall pipeline.

## Solution 

I will approach this project as an evaluation pipeline design problem rather than a simple modeling exercise. The goal is to build a workflow that produces fair, reproducible, and interpretable results.

My plan is to start from the theoretical foundations, prepare the data carefully, implement and validate the splitting logic, compare it against sklearn, and then extend the pipeline with feature selection and hyperparameter optimization before making a final decision.

## Roadmap

This notebook is organized into the following sections:

1. Theory Questions  
2. Data Preparation  
3. Custom Split Methods  
4. Custom Cross-Validation Methods  
5. Validation Comparison  
6. Feature Selection  
7. Hyperparameter Optimization  
8. Final Comparison and Conclusion

# 1. Theory Questions

Before moving into the implementation, I want to clarify the main concepts behind this project. Since the whole notebook is built around validation, feature selection, and hyperparameter tuning, it makes sense to define these ideas first in a simple and practical way.

## 1.1 What is Leave-One-Out Cross-Validation?

Leave-One-Out Cross-Validation (LOOCV) is a special case of cross-validation where each sample is used once as the test set, while all the remaining samples are used for training. If the dataset has \(N\) samples, then the model is trained \(N\) times. Each time, one sample is left out for testing and the rest are used for training.

What makes LOOCV interesting is that it uses almost the entire dataset for training in every iteration, which can be useful when the dataset is very small. At the same time, this also makes it expensive, because the model has to be retrained once per sample. Another issue is that each test fold contains only one observation, so the final estimate can be noisy and unstable.

In short, LOOCV is conceptually simple and useful for small datasets, but in practice it is often too expensive or too sensitive compared with standard K-Fold cross-validation.

## 1.2 How do Grid Search, Randomized Search, and Bayesian Optimization work?

Hyperparameter optimization is the process of searching for the best model settings before training. These settings are not learned by the model itself, so they have to be selected externally.

### Grid Search

Grid Search is the most direct method. I define a list of candidate values for each hyperparameter, then I try every possible combination. This makes Grid Search easy to understand and very systematic, but it can become slow when the search space is large.

The main advantage of Grid Search is that it is exhaustive inside the chosen grid. The downside is that it can waste a lot of time evaluating combinations that are not promising.

### Randomized Search

Randomized Search works in a similar way, but instead of trying every combination, it randomly samples only a limited number of combinations. This makes it faster and often more practical when the search space is large.

Its main strength is efficiency. With fewer trials, it can still explore a broad range of values. Its main weakness is that it may miss the best combination because it does not check the whole grid.

### Bayesian Optimization

Bayesian Optimization is a more adaptive strategy. Instead of exploring combinations blindly, it uses the results from previous trials to decide which hyperparameter values should be tested next. In other words, it tries to learn which parts of the search space look promising and focuses the search there.

This makes Bayesian Optimization more efficient than brute-force methods in many cases, especially when training is expensive. The trade-off is that it is more complex and less straightforward than Grid Search or Randomized Search.

In practice, I can think about the three methods like this:
- Grid Search checks everything in a fixed search space.
- Randomized Search checks only a random subset.
- Bayesian Optimization tries to search more intelligently.

## 1.3 Feature Selection Methods

Feature selection is the process of choosing the most useful features for the model instead of using everything by default. This matters because not every feature contains useful signal. Some features are redundant, some are noisy, and some can even hurt performance. Good feature selection can improve model quality, reduce overfitting, and make the model easier to interpret.

Feature selection methods are usually divided into three broad groups:

### Filter methods

Filter methods evaluate features using statistical criteria before model training. They are usually fast and simple. They do not depend heavily on a specific model, which makes them useful as an early filtering step.

### Wrapper methods

Wrapper methods evaluate subsets of features by repeatedly training a model and comparing the results. They are usually more expensive than filter methods, but they can capture interactions between features better.

### Embedded methods

Embedded methods perform feature selection during model training itself. In this case, the model is not only learning how to predict, but also helping identify which features are important.

## 1.4 Pearson Correlation

Pearson correlation measures the strength of a linear relationship between a feature and the target. A high positive value means that the target tends to increase when the feature increases. A high negative value means the opposite. A value close to zero means there is little or no linear relationship.

Pearson correlation is simple and useful as a fast filter method, but it only captures linear relationships. If the relationship is non-linear, Pearson may underestimate the importance of a feature.

## 1.5 Chi-Square

Chi-square is often used to measure dependence between categorical variables. In feature selection, it is usually applied when the feature and the target can be treated as categorical or discrete. The idea is to test whether the observed relationship between them is stronger than what would be expected by chance.

Chi-square is useful in classification-style settings or when dealing with discrete variables, but it is not a general-purpose method for every regression problem.

## 1.6 Lasso

Lasso is a linear model with L1 regularization. What makes it especially useful for feature selection is that it can shrink some coefficients all the way to zero. Once a coefficient becomes zero, that feature is effectively removed from the model.

This makes Lasso an embedded feature selection method. It is attractive because it performs prediction and selection at the same time. The limitation is that its behavior depends on scaling and on the regularization strength, so the features usually need to be normalized before using it.

## 1.7 Permutation Importance

Permutation importance measures how much the model depends on a feature after training. The basic idea is simple: if I randomly shuffle the values of one feature and the model performance drops noticeably, then that feature was important. If performance barely changes, then the feature was probably not very useful.

This method is practical and intuitive because it evaluates importance based on real impact on performance, not just on coefficients or correlations. However, it depends on the trained model and can be more expensive than simple filter methods.

## 1.8 SHAP

SHAP is a feature importance and explanation framework that tries to explain how each feature contributes to individual predictions. Instead of saying only that a feature is globally important, SHAP also helps show how it pushes a specific prediction up or down.

This makes SHAP especially useful when I want interpretability, not just ranking. It is one of the strongest tools for explanation, but it is also heavier and more complex than simple methods like correlation or permutation importance.

## Final note

The main reason these concepts matter in this project is that validation, feature selection, and hyperparameter optimization are deeply connected. If they are done carelessly, the final score can become misleading. If they are done correctly, they help build a much more reliable and fair evaluation pipeline.

# 2. Data Preparation

Before moving to the validation part, I need to prepare the dataset in a usable form. Here, I will inspect the raw data, clean the `features` column, and create the required binary features that will be used in the next steps.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter

df = pd.read_json("data/train.json")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData info:")
print(df.info())

print("\nFirst rows:")
display(df.head())

print("\nSample values from the 'features' column:")
display(df["features"].head())

Shape: (49352, 15)

Columns:
['bathrooms', 'bedrooms', 'building_id', 'created', 'description', 'display_address', 'features', 'latitude', 'listing_id', 'longitude', 'manager_id', 'photos', 'price', 'street_address', 'interest_level']

Data info:
<class 'pandas.core.frame.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        49352 non-null  float64
 1   bedrooms         49352 non-null  int64  
 2   building_id      49352 non-null  object 
 3   created          49352 non-null  object 
 4   description      49352 non-null  object 
 5   display_address  49352 non-null  object 
 6   features         49352 non-null  object 
 7   latitude         49352 non-null  float64
 8   listing_id       49352 non-null  int64  
 9   longitude        49352 non-null  float64
 10  manager_id       49352 non-null  object 
 11  photos           49352 non-null  object 
 12 

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low



Sample values from the 'features' column:


4     [Dining Room, Pre-War, Laundry in Building, Di...
6     [Doorman, Elevator, Laundry in Building, Dishw...
9     [Doorman, Elevator, Laundry in Building, Laund...
10                                                   []
15    [Doorman, Elevator, Fitness Center, Laundry in...
Name: features, dtype: object

## Creating the required binary features

The `features` column contains raw apartment attributes stored as text lists. To use them in modeling, I need to normalize their names first, then convert the required values into binary columns.

In [2]:
required_features = [
    'Elevator',
    'HardwoodFloors',
    'CatsAllowed',
    'DogsAllowed',
    'Doorman',
    'Dishwasher',
    'NoFee',
    'LaundryinBuilding',
    'FitnessCenter',
    'Pre-War',
    'LaundryinUnit',
    'RoofDeck',
    'OutdoorSpace',
    'DiningRoom',
    'HighSpeedInternet',
    'Balcony',
    'SwimmingPool',
    'LaundryInBuilding',
    'NewConstruction',
    'Terrace'
]

def normalize_feature_name(feature):
    return str(feature).replace("-", "").replace(" ", "").strip()

df["features_clean"] = df["features"].apply(
    lambda values: [normalize_feature_name(v) for v in values]
)

for feature in required_features:
    feature_norm = normalize_feature_name(feature)
    df[feature] = df["features_clean"].apply(lambda values: int(feature_norm in values))

feature_list = ["bathrooms", "bedrooms"] + required_features
model_df = df[feature_list + ["price"]].copy()

print("Number of required binary features:", len(required_features))
print("Number of modeling features:", len(feature_list))
print("\nPreview of created binary features:")
display(df[required_features].head())

print("\nSample of original vs cleaned features:")
display(df[["features", "features_clean"]].head())

print("\nModel dataframe shape:", model_df.shape)

Number of required binary features: 20
Number of modeling features: 22

Preview of created binary features:


,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,0,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0
6,1,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,1,1,0,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0
10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0



Sample of original vs cleaned features:


,features,features_clean
4,"[Dining Room, Pre-War, Laundry in Building, Di...","[DiningRoom, PreWar, LaundryinBuilding, Dishwa..."
6,"[Doorman, Elevator, Laundry in Building, Dishw...","[Doorman, Elevator, LaundryinBuilding, Dishwas..."
9,"[Doorman, Elevator, Laundry in Building, Laund...","[Doorman, Elevator, LaundryinBuilding, Laundry..."
10,[],[]
15,"[Doorman, Elevator, Fitness Center, Laundry in...","[Doorman, Elevator, FitnessCenter, LaundryinBu..."



Model dataframe shape: (49352, 23)


# 3. Custom Split Methods

In this section, I will implement the required data splitting methods manually. Since some of them depend on time, I will first convert the `created` column into a proper datetime format.

In [3]:
df["created"] = pd.to_datetime(df["created"])

print(df["created"].dtype)
display(df[["created"]].head())

datetime64[ns]


,created
4,2016-06-16 05:55:27
6,2016-06-01 05:44:33
9,2016-06-14 15:19:59
10,2016-06-24 07:54:24
15,2016-06-28 03:50:23


## 3.1 Random train/test split

I will start with the simplest split method: a random train/test split controlled by `test_size`. This method is appropriate when the data does not need time-aware splitting.

In [4]:
def random_train_test_split(df, test_size=0.2, random_state=42):
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1")

    np.random.seed(random_state)

    indices = np.arange(len(df))
    np.random.shuffle(indices)

    test_count = int(len(df) * test_size)

    test_indices = indices[:test_count]
    train_indices = indices[test_count:]

    train_df = df.iloc[train_indices].copy()
    test_df = df.iloc[test_indices].copy()

    return train_df, test_df


train_df, test_df = random_train_test_split(model_df, test_size=0.2, random_state=42)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Total rows preserved:", len(train_df) + len(test_df) == len(model_df))
print("No overlap between train and test:", len(set(train_df.index).intersection(set(test_df.index))) == 0)

Train shape: (39482, 23)
Test shape: (9870, 23)
Total rows preserved: True
No overlap between train and test: True


## 3.2 Random train/validation/test split

The next step is to split the dataset into three random parts: training, validation, and test. This is useful when I want to keep a separate validation set for model selection, feature selection, or hyperparameter tuning.

In [5]:
def random_train_val_test_split(df, validation_size=0.2, test_size=0.2, random_state=42):
    if not 0 < validation_size < 1:
        raise ValueError("validation_size must be between 0 and 1")
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1")
    if validation_size + test_size >= 1:
        raise ValueError("validation_size + test_size must be less than 1")

    np.random.seed(random_state)

    indices = np.arange(len(df))
    np.random.shuffle(indices)

    test_count = int(len(df) * test_size)
    val_count = int(len(df) * validation_size)

    test_indices = indices[:test_count]
    val_indices = indices[test_count:test_count + val_count]
    train_indices = indices[test_count + val_count:]

    train_df = df.iloc[train_indices].copy()
    val_df = df.iloc[val_indices].copy()
    test_df = df.iloc[test_indices].copy()

    return train_df, val_df, test_df


train_df_3, val_df_3, test_df_3 = random_train_val_test_split(
    model_df,
    validation_size=0.2,
    test_size=0.2,
    random_state=42
)

print("Train shape:", train_df_3.shape)
print("Validation shape:", val_df_3.shape)
print("Test shape:", test_df_3.shape)

total_ok = len(train_df_3) + len(val_df_3) + len(test_df_3) == len(model_df)
print("Total rows preserved:", total_ok)

all_overlap = (
    len(set(train_df_3.index).intersection(set(val_df_3.index))) == 0 and
    len(set(train_df_3.index).intersection(set(test_df_3.index))) == 0 and
    len(set(val_df_3.index).intersection(set(test_df_3.index))) == 0
)
print("No overlap across splits:", all_overlap)

Train shape: (29612, 23)
Validation shape: (9870, 23)
Test shape: (9870, 23)
Total rows preserved: True
No overlap across splits: True


## 3.3 Date-based train/test split

Now I will implement a time-aware train/test split using a date threshold. This method is more appropriate when the data has a time relationship and I want to avoid using future information during training.

In [6]:
def date_train_test_split(df, date_col="created", date_split="2016-06-15"):
    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found in dataframe")

    split_date = pd.to_datetime(date_split)

    train_df = df[df[date_col] < split_date].copy()
    test_df = df[df[date_col] >= split_date].copy()

    return train_df, test_df


train_time_df, test_time_df = date_train_test_split(df, date_col="created", date_split="2016-06-15")

print("Train shape:", train_time_df.shape)
print("Test shape:", test_time_df.shape)

print("Total rows preserved:", len(train_time_df) + len(test_time_df) == len(df))
print("Latest train date:", train_time_df["created"].max())
print("Earliest test date:", test_time_df["created"].min())
print("Time order respected:", train_time_df["created"].max() < test_time_df["created"].min())

Train shape: (40991, 36)
Test shape: (8361, 36)
Total rows preserved: True
Latest train date: 2016-06-14 22:51:35
Earliest test date: 2016-06-15 01:10:37
Time order respected: True


## 3.4 Date-based train/validation/test split

Here I extend the time-aware split into three parts: training, validation, and test. This allows me to keep the whole evaluation process consistent with the time order of the data.

In [7]:
def date_train_val_test_split(df, date_col="created", validation_date="2016-06-10", test_date="2016-06-20"):
    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found in dataframe")

    validation_date = pd.to_datetime(validation_date)
    test_date = pd.to_datetime(test_date)

    if validation_date >= test_date:
        raise ValueError("validation_date must be earlier than test_date")

    train_df = df[df[date_col] < validation_date].copy()
    val_df = df[(df[date_col] >= validation_date) & (df[date_col] < test_date)].copy()
    test_df = df[df[date_col] >= test_date].copy()

    return train_df, val_df, test_df


train_time_3, val_time_3, test_time_3 = date_train_val_test_split(
    df,
    date_col="created",
    validation_date="2016-06-10",
    test_date="2016-06-20"
)

print("Train shape:", train_time_3.shape)
print("Validation shape:", val_time_3.shape)
print("Test shape:", test_time_3.shape)

print("Total rows preserved:", len(train_time_3) + len(val_time_3) + len(test_time_3) == len(df))

no_overlap = (
    len(set(train_time_3.index).intersection(set(val_time_3.index))) == 0 and
    len(set(train_time_3.index).intersection(set(test_time_3.index))) == 0 and
    len(set(val_time_3.index).intersection(set(test_time_3.index))) == 0
)
print("No overlap across splits:", no_overlap)

print("Latest train date:", train_time_3["created"].max())
print("Earliest validation date:", val_time_3["created"].min())
print("Latest validation date:", val_time_3["created"].max())
print("Earliest test date:", test_time_3["created"].min())

time_order_ok = (
    train_time_3["created"].max() < val_time_3["created"].min() and
    val_time_3["created"].max() < test_time_3["created"].min()
)
print("Time order respected:", time_order_ok)

Train shape: (37793, 36)
Validation shape: (5563, 36)
Test shape: (5996, 36)
Total rows preserved: True
No overlap across splits: True
Latest train date: 2016-06-09 23:42:21
Earliest validation date: 2016-06-10 01:01:13
Latest validation date: 2016-06-19 21:57:21
Earliest test date: 2016-06-20 10:00:03
Time order respected: True


## 3.5 Deterministic splitting

A deterministic split means that the same input data and the same configuration should always produce the same split.

In the random split functions, I made the procedure deterministic by using a fixed `random_state`. This is important because it makes the results reproducible and allows fair comparisons between different models or experiments.

For the date-based splits, the procedure is already deterministic because the split depends directly on fixed date thresholds rather than random sampling.

# 4. Custom Cross-Validation Methods

After implementing the basic split methods, the next step is to implement cross-validation methods manually. Unlike a single holdout split, cross-validation evaluates the model across multiple splits, which usually gives a more stable view of model performance.

## 4.1 K-Fold

In [8]:
def custom_kfold_indices(df, k=5, random_state=42, shuffle=True):
    if k < 2:
        raise ValueError("k must be at least 2")

    indices = np.arange(len(df))

    if shuffle:
        np.random.seed(random_state)
        np.random.shuffle(indices)

    folds = np.array_split(indices, k)
    split_indices = []

    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        split_indices.append((train_idx, test_idx))

    return split_indices


kfold_splits = custom_kfold_indices(model_df, k=5, random_state=42, shuffle=True)

print("Number of folds:", len(kfold_splits))

for i, (train_idx, test_idx) in enumerate(kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Number of folds: 5
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870


## 4.2 Grouped K-Fold

In [9]:
def custom_group_kfold_indices(df, group_field, k=5, random_state=42, shuffle=True):
    if k < 2:
        raise ValueError("k must be at least 2")
    if group_field not in df.columns:
        raise ValueError(f"Column '{group_field}' not found in dataframe")

    unique_groups = df[group_field].dropna().unique()

    if shuffle:
        np.random.seed(random_state)
        np.random.shuffle(unique_groups)

    group_folds = np.array_split(unique_groups, k)
    split_indices = []

    for i in range(k):
        test_groups = set(group_folds[i])
        test_mask = df[group_field].isin(test_groups)
        train_mask = ~test_mask

        train_idx = df[train_mask].index.to_numpy()
        test_idx = df[test_mask].index.to_numpy()

        split_indices.append((train_idx, test_idx))

    return split_indices


group_kfold_splits = custom_group_kfold_indices(df, group_field="manager_id", k=5, random_state=42, shuffle=True)

print("Number of folds:", len(group_kfold_splits))

for i, (train_idx, test_idx) in enumerate(group_kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Number of folds: 5
Fold 1: train=39400, test=9952
Fold 2: train=36872, test=12480
Fold 3: train=40065, test=9287
Fold 4: train=40795, test=8557
Fold 5: train=40276, test=9076


In [10]:
for i, (train_idx, test_idx) in enumerate(group_kfold_splits, start=1):
    train_groups = set(df.loc[train_idx, "manager_id"])
    test_groups = set(df.loc[test_idx, "manager_id"])
    overlap = train_groups.intersection(test_groups)

    print(f"Fold {i}: overlapping groups = {len(overlap)}")

Fold 1: overlapping groups = 0
Fold 2: overlapping groups = 0
Fold 3: overlapping groups = 0
Fold 4: overlapping groups = 0
Fold 5: overlapping groups = 0


## 4.3 Stratified K-Fold

In [11]:
def custom_stratified_kfold_indices(df, stratify_field, k=5, random_state=42, shuffle=True):
    if k < 2:
        raise ValueError("k must be at least 2")
    if stratify_field not in df.columns:
        raise ValueError(f"Column '{stratify_field}' not found in dataframe")

    indices = np.arange(len(df))
    labels = df[stratify_field].to_numpy()

    if shuffle:
        np.random.seed(random_state)

    split_indices = []
    test_folds = [[] for _ in range(k)]

    for label in np.unique(labels):
        label_indices = indices[labels == label]

        if shuffle:
            np.random.shuffle(label_indices)

        label_folds = np.array_split(label_indices, k)

        for i in range(k):
            test_folds[i].extend(label_folds[i])

    for i in range(k):
        test_idx = np.array(test_folds[i])
        train_idx = np.setdiff1d(indices, test_idx)
        split_indices.append((train_idx, test_idx))

    return split_indices


df["price_bin"] = pd.qcut(df["price"], q=5, labels=False, duplicates="drop")

stratified_splits = custom_stratified_kfold_indices(
    df,
    stratify_field="price_bin",
    k=5,
    random_state=42,
    shuffle=True
)

print("Number of folds:", len(stratified_splits))

for i, (train_idx, test_idx) in enumerate(stratified_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Number of folds: 5
Fold 1: train=39479, test=9873
Fold 2: train=39480, test=9872
Fold 3: train=39482, test=9870
Fold 4: train=39483, test=9869
Fold 5: train=39484, test=9868


In [12]:
for i, (_, test_idx) in enumerate(stratified_splits, start=1):
    distribution = df.iloc[test_idx]["price_bin"].value_counts(normalize=True).sort_index()
    print(f"\nFold {i} price_bin distribution:")
    print(distribution)


Fold 1 price_bin distribution:
price_bin
0    0.202471
1    0.199129
2    0.202168
3    0.201459
4    0.194774
Name: proportion, dtype: float64

Fold 2 price_bin distribution:
price_bin
0    0.202492
1    0.199149
2    0.202087
3    0.201479
4    0.194793
Name: proportion, dtype: float64

Fold 3 price_bin distribution:
price_bin
0    0.202533
1    0.199088
2    0.202128
3    0.201520
4    0.194732
Name: proportion, dtype: float64

Fold 4 price_bin distribution:
price_bin
0    0.202553
1    0.199108
2    0.202148
3    0.201439
4    0.194751
Name: proportion, dtype: float64

Fold 5 price_bin distribution:
price_bin
0    0.202473
1    0.199128
2    0.202169
3    0.201459
4    0.194771
Name: proportion, dtype: float64


## 4.4 Time Series Split

In [13]:
def custom_time_series_split_indices(df, date_field, k=5):
    if k < 2:
        raise ValueError("k must be at least 2")
    if date_field not in df.columns:
        raise ValueError(f"Column '{date_field}' not found in dataframe")

    df_sorted = df.sort_values(date_field).reset_index(drop=False)
    n = len(df_sorted)

    fold_size = n // k
    split_indices = []

    for i in range(1, k):
        train_end = i * fold_size
        test_end = (i + 1) * fold_size if i < k - 1 else n

        train_idx = df_sorted.loc[:train_end - 1, "index"].to_numpy()
        test_idx = df_sorted.loc[train_end:test_end - 1, "index"].to_numpy()

        split_indices.append((train_idx, test_idx))

    return split_indices


time_splits = custom_time_series_split_indices(df, date_field="created", k=5)

print("Number of splits:", len(time_splits))

for i, (train_idx, test_idx) in enumerate(time_splits, start=1):
    print(f"Split {i}: train={len(train_idx)}, test={len(test_idx)}")

Number of splits: 4
Split 1: train=9870, test=9870
Split 2: train=19740, test=9870
Split 3: train=29610, test=9870
Split 4: train=39480, test=9872


In [14]:
for i, (train_idx, test_idx) in enumerate(time_splits, start=1):
    train_dates = df.loc[train_idx, "created"]
    test_dates = df.loc[test_idx, "created"]

    print(f"\nSplit {i}")
    print("Latest train date:", train_dates.max())
    print("Earliest test date:", test_dates.min())
    print("Time order respected:", train_dates.max() < test_dates.min())


Split 1
Latest train date: 2016-04-19 03:56:39
Earliest test date: 2016-04-19 03:57:40
Time order respected: True

Split 2
Latest train date: 2016-05-07 03:11:17
Earliest test date: 2016-05-07 03:11:44
Time order respected: True

Split 3
Latest train date: 2016-05-24 22:30:51
Earliest test date: 2016-05-25 01:10:42
Time order respected: True

Split 4
Latest train date: 2016-06-12 08:16:40
Earliest test date: 2016-06-12 08:17:43
Time order respected: True


# 5. Validation Comparison

After implementing the validation methods manually, I now want to compare them with the equivalent sklearn implementations. The goal here is not only to check correctness, but also to understand how different validation strategies affect the data split and the final evaluation logic.

In [15]:
from sklearn.model_selection import KFold

sklearn_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
sklearn_kfold_splits = list(sklearn_kfold.split(model_df))

print("Custom K-Fold:")
for i, (train_idx, test_idx) in enumerate(kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

print("\nSklearn K-Fold:")
for i, (train_idx, test_idx) in enumerate(sklearn_kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Custom K-Fold:
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870

Sklearn K-Fold:
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870


In [16]:
custom_train_idx, custom_test_idx = kfold_splits[0]
sklearn_train_idx, sklearn_test_idx = sklearn_kfold_splits[0]

selected_features = ["bathrooms", "bedrooms", "Elevator", "Doorman", "Dishwasher"]

custom_train_stats = model_df.iloc[custom_train_idx][selected_features].mean().rename("custom_train_mean")
sklearn_train_stats = model_df.iloc[sklearn_train_idx][selected_features].mean().rename("sklearn_train_mean")

comparison_df = pd.concat([custom_train_stats, sklearn_train_stats], axis=1)
comparison_df["absolute_diff"] = (comparison_df["custom_train_mean"] - comparison_df["sklearn_train_mean"]).abs()

comparison_df

,custom_train_mean,sklearn_train_mean,absolute_diff
bathrooms,1.212773,1.212773,0.0
bedrooms,1.538462,1.538462,0.0
Elevator,0.525493,0.525493,0.0
Doorman,0.424407,0.424407,0.0
Dishwasher,0.413085,0.413085,0.0


The comparison shows that the custom K-Fold implementation produces the same training distribution as sklearn for the selected features in the first fold. This supports the correctness of the custom implementation, not only in terms of fold sizes, but also in terms of feature distribution inside the training split.

In [17]:
from sklearn.model_selection import GroupKFold

sklearn_group_kfold = GroupKFold(n_splits=5)
sklearn_group_splits = list(sklearn_group_kfold.split(df, groups=df["manager_id"]))

print("Custom Group K-Fold:")
for i, (train_idx, test_idx) in enumerate(group_kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

print("\nSklearn Group K-Fold:")
for i, (train_idx, test_idx) in enumerate(sklearn_group_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Custom Group K-Fold:
Fold 1: train=39400, test=9952
Fold 2: train=36872, test=12480
Fold 3: train=40065, test=9287
Fold 4: train=40795, test=8557
Fold 5: train=40276, test=9076

Sklearn Group K-Fold:
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870


In [18]:
for i, ((custom_train_idx, custom_test_idx), (sk_train_idx, sk_test_idx)) in enumerate(
    zip(group_kfold_splits, sklearn_group_splits), start=1
):
    custom_overlap = set(df.loc[custom_train_idx, "manager_id"]).intersection(set(df.loc[custom_test_idx, "manager_id"]))
    sklearn_overlap = set(df.iloc[sk_train_idx]["manager_id"]).intersection(set(df.iloc[sk_test_idx]["manager_id"]))

    print(f"Fold {i}: custom_overlap={len(custom_overlap)}, sklearn_overlap={len(sklearn_overlap)}")

Fold 1: custom_overlap=0, sklearn_overlap=0
Fold 2: custom_overlap=0, sklearn_overlap=0
Fold 3: custom_overlap=0, sklearn_overlap=0
Fold 4: custom_overlap=0, sklearn_overlap=0
Fold 5: custom_overlap=0, sklearn_overlap=0


The custom and sklearn implementations both satisfy the main goal of Group K-Fold: the same group does not appear in both training and test sets within the same fold.

However, the fold sizes are more uneven in the custom implementation. This happens because my version splits the groups directly, while sklearn uses a more balanced strategy to keep the number of samples per fold closer across the splits.

In [19]:
from sklearn.model_selection import StratifiedKFold

sklearn_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sklearn_stratified_splits = list(sklearn_stratified.split(df, df["price_bin"]))

print("Custom Stratified K-Fold:")
for i, (train_idx, test_idx) in enumerate(stratified_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

print("\nSklearn Stratified K-Fold:")
for i, (train_idx, test_idx) in enumerate(sklearn_stratified_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Custom Stratified K-Fold:
Fold 1: train=39479, test=9873
Fold 2: train=39480, test=9872
Fold 3: train=39482, test=9870
Fold 4: train=39483, test=9869
Fold 5: train=39484, test=9868

Sklearn Stratified K-Fold:
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870


In [20]:
print("Custom Stratified K-Fold distributions:")
for i, (_, test_idx) in enumerate(stratified_splits, start=1):
    distribution = df.iloc[test_idx]["price_bin"].value_counts(normalize=True).sort_index()
    print(f"\nFold {i}")
    print(distribution)

print("\nSklearn Stratified K-Fold distributions:")
for i, (_, test_idx) in enumerate(sklearn_stratified_splits, start=1):
    distribution = df.iloc[test_idx]["price_bin"].value_counts(normalize=True).sort_index()
    print(f"\nFold {i}")
    print(distribution)

Custom Stratified K-Fold distributions:

Fold 1
price_bin
0    0.202471
1    0.199129
2    0.202168
3    0.201459
4    0.194774
Name: proportion, dtype: float64

Fold 2
price_bin
0    0.202492
1    0.199149
2    0.202087
3    0.201479
4    0.194793
Name: proportion, dtype: float64

Fold 3
price_bin
0    0.202533
1    0.199088
2    0.202128
3    0.201520
4    0.194732
Name: proportion, dtype: float64

Fold 4
price_bin
0    0.202553
1    0.199108
2    0.202148
3    0.201439
4    0.194751
Name: proportion, dtype: float64

Fold 5
price_bin
0    0.202473
1    0.199128
2    0.202169
3    0.201459
4    0.194771
Name: proportion, dtype: float64

Sklearn Stratified K-Fold distributions:

Fold 1
price_bin
0    0.202512
1    0.199169
2    0.202208
3    0.201398
4    0.194712
Name: proportion, dtype: float64

Fold 2
price_bin
0    0.202512
1    0.199169
2    0.202107
3    0.201398
4    0.194813
Name: proportion, dtype: float64

Fold 3
price_bin
0    0.202432
1    0.199088
2    0.202128
3    0.2015

The custom Stratified K-Fold implementation preserves the target distribution across folds very well. The resulting distributions are also very close to the sklearn implementation, which supports the correctness of the custom method both conceptually and practically.

## Choosing the most appropriate validation scheme

After comparing the implemented validation methods, I do not think there is a single universally best scheme for every situation. The best choice depends on the structure of the data and the type of leakage I want to avoid.

- Standard K-Fold is a strong general-purpose choice when the observations are independent.
- Group K-Fold is more appropriate when the same entity may appear multiple times and group leakage is a risk.
- Stratified K-Fold is useful when I want to preserve the target distribution across folds.
- Time-based splitting and TimeSeriesSplit are the most appropriate choices when the data has a time relationship.

For this dataset, since the `created` column is available and time-aware splitting is part of the project requirements, I consider time-based validation to be the most realistic option when the goal is to simulate real future prediction. It respects the chronological structure of the data and avoids using future information during training.

# 6. Feature Selection

In this section, I will compare different feature selection methods. I will start with Lasso, since it can shrink some coefficients toward zero and can therefore be used as an embedded feature selection method.

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error

train_df_fs, val_df_fs, test_df_fs = random_train_val_test_split(
    model_df,
    validation_size=0.2,
    test_size=0.2,
    random_state=42
)

X_train = train_df_fs[feature_list]
y_train = train_df_fs["price"]

X_val = val_df_fs[feature_list]
y_val = val_df_fs["price"]

X_test = test_df_fs[feature_list]
y_test = test_df_fs["price"]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

lasso = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso.fit(X_train_scaled, y_train)

val_pred = lasso.predict(X_val_scaled)

print("Validation MAE:", mean_absolute_error(y_val, val_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Validation MAE: 1048.6533516489706
Validation MSE: 121410803.13701078


In [22]:
lasso_importance = pd.Series(np.abs(lasso.coef_), index=feature_list).sort_values(ascending=False)

print("Lasso feature importance ranking:")
display(lasso_importance)

top_10_lasso_features = lasso_importance.head(10).index.tolist()

print("Top 10 features selected by Lasso:")
print(top_10_lasso_features)

Lasso feature importance ranking:


bathrooms            1197.597464
Doorman               579.090377
bedrooms              508.648988
LaundryinUnit         215.942613
DogsAllowed           174.728384
LaundryinBuilding     168.910883
NoFee                 129.676525
HardwoodFloors        122.627942
LaundryInBuilding     117.983823
HighSpeedInternet      88.465794
Elevator               77.200918
Terrace                67.721138
CatsAllowed            62.593183
DiningRoom             51.153462
FitnessCenter          48.421051
OutdoorSpace           44.669258
Pre-War                40.402450
Balcony                28.945015
RoofDeck               25.793350
Dishwasher             15.312262
NewConstruction        10.144654
SwimmingPool            3.158979
dtype: float64

Top 10 features selected by Lasso:
['bathrooms', 'Doorman', 'bedrooms', 'LaundryinUnit', 'DogsAllowed', 'LaundryinBuilding', 'NoFee', 'HardwoodFloors', 'LaundryInBuilding', 'HighSpeedInternet']


In [23]:
X_train_top10 = train_df_fs[top_10_lasso_features]
X_val_top10 = val_df_fs[top_10_lasso_features]
X_test_top10 = test_df_fs[top_10_lasso_features]

scaler_top10 = StandardScaler()

X_train_top10_scaled = scaler_top10.fit_transform(X_train_top10)
X_val_top10_scaled = scaler_top10.transform(X_val_top10)
X_test_top10_scaled = scaler_top10.transform(X_test_top10)

lasso_top10 = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_top10.fit(X_train_top10_scaled, y_train)

val_pred_top10 = lasso_top10.predict(X_val_top10_scaled)

print("Validation MAE with top 10 Lasso features:", mean_absolute_error(y_val, val_pred_top10))
print("Validation MSE with top 10 Lasso features:", mean_squared_error(y_val, val_pred_top10))

Validation MAE with top 10 Lasso features: 1047.6416572587837
Validation MSE with top 10 Lasso features: 121438828.93016015


At this point, I compare the validation performance of the Lasso model using all features against the same model trained only on the top 10 features selected by coefficient magnitude. This helps me check whether a smaller feature subset can preserve most of the useful signal.

The top 10 features selected by Lasso preserved almost the same validation performance as the full feature set. The MAE improved slightly, while the MSE became slightly worse, but the difference was very small overall.

This suggests that the reduced feature subset retains most of the predictive signal, which makes it a useful and more compact representation of the data.

In [24]:
nan_ratio = train_df_fs[feature_list].isna().mean()

correlation_with_target = train_df_fs[feature_list].corrwith(train_df_fs["price"]).abs()

simple_selection_df = pd.DataFrame({
    "nan_ratio": nan_ratio,
    "abs_correlation": correlation_with_target
}).sort_values(by=["nan_ratio", "abs_correlation"], ascending=[True, False])

display(simple_selection_df)

top_10_simple_features = simple_selection_df.head(10).index.tolist()

print("Top 10 features selected by nan-ratio + correlation:")
print(top_10_simple_features)

,nan_ratio,abs_correlation
bathrooms,0.0,0.167535
bedrooms,0.0,0.119073
Doorman,0.0,0.069305
LaundryinUnit,0.0,0.056649
DiningRoom,0.0,0.050734
Elevator,0.0,0.043884
FitnessCenter,0.0,0.041395
Dishwasher,0.0,0.039359
Terrace,0.0,0.034213
OutdoorSpace,0.0,0.031209


Top 10 features selected by nan-ratio + correlation:
['bathrooms', 'bedrooms', 'Doorman', 'LaundryinUnit', 'DiningRoom', 'Elevator', 'FitnessCenter', 'Dishwasher', 'Terrace', 'OutdoorSpace']


In [25]:
X_train_simple = train_df_fs[top_10_simple_features]
X_val_simple = val_df_fs[top_10_simple_features]

scaler_simple = StandardScaler()

X_train_simple_scaled = scaler_simple.fit_transform(X_train_simple)
X_val_simple_scaled = scaler_simple.transform(X_val_simple)

lasso_simple = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_simple.fit(X_train_simple_scaled, y_train)

val_pred_simple = lasso_simple.predict(X_val_simple_scaled)

print("Validation MAE with nan-ratio + correlation top 10:", mean_absolute_error(y_val, val_pred_simple))
print("Validation MSE with nan-ratio + correlation top 10:", mean_squared_error(y_val, val_pred_simple))

Validation MAE with nan-ratio + correlation top 10: 1046.8935074623876
Validation MSE with nan-ratio + correlation top 10: 121520160.48300299


The nan-ratio component did not have a real effect here because all selected modeling features had zero missing-value ratio. As a result, this method was effectively driven by absolute correlation with the target.

The results were still competitive. The MAE was slightly better than the Lasso top-10 result, while the MSE was slightly worse. Overall, the difference remained small, which suggests that a simple correlation-based filter can still produce a useful compact feature subset.

## 6.3 Permutation Importance

Next, I will use permutation importance. The main idea here is to measure feature importance by checking how much the validation performance drops after shuffling one feature at a time.

In [26]:
def permutation_importance_mae(model, X_val_scaled, y_val, feature_names, random_state=42):
    np.random.seed(random_state)

    baseline_pred = model.predict(X_val_scaled)
    baseline_mae = mean_absolute_error(y_val, baseline_pred)

    importance_scores = {}

    for i, feature in enumerate(feature_names):
        X_permuted = X_val_scaled.copy()
        shuffled_values = X_permuted[:, i].copy()
        np.random.shuffle(shuffled_values)
        X_permuted[:, i] = shuffled_values

        permuted_pred = model.predict(X_permuted)
        permuted_mae = mean_absolute_error(y_val, permuted_pred)

        importance_scores[feature] = permuted_mae - baseline_mae

    return pd.Series(importance_scores).sort_values(ascending=False)


perm_importance = permutation_importance_mae(
    lasso,
    X_val_scaled,
    y_val,
    feature_list,
    random_state=42
)

print("Permutation importance ranking:")
display(perm_importance)

top_10_perm_features = perm_importance.head(10).index.tolist()

print("Top 10 features selected by permutation importance:")
print(top_10_perm_features)

Permutation importance ranking:


bathrooms            414.083950
Doorman              198.057688
bedrooms             164.419857
LaundryinUnit         34.403880
LaundryinBuilding     15.901160
DogsAllowed           11.555790
LaundryInBuilding      9.827116
CatsAllowed            8.609367
NoFee                  7.682270
HighSpeedInternet      5.829012
Elevator               3.040821
Terrace                2.178411
HardwoodFloors         1.809259
FitnessCenter          1.636550
DiningRoom             1.593211
OutdoorSpace           1.541056
Dishwasher             1.456412
Balcony                1.116229
RoofDeck               0.483776
NewConstruction        0.128844
Pre-War                0.014787
SwimmingPool           0.004346
dtype: float64

Top 10 features selected by permutation importance:
['bathrooms', 'Doorman', 'bedrooms', 'LaundryinUnit', 'LaundryinBuilding', 'DogsAllowed', 'LaundryInBuilding', 'CatsAllowed', 'NoFee', 'HighSpeedInternet']


In [27]:
X_train_perm = train_df_fs[top_10_perm_features]
X_val_perm = val_df_fs[top_10_perm_features]

scaler_perm = StandardScaler()

X_train_perm_scaled = scaler_perm.fit_transform(X_train_perm)
X_val_perm_scaled = scaler_perm.transform(X_val_perm)

lasso_perm = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_perm.fit(X_train_perm_scaled, y_train)

val_pred_perm = lasso_perm.predict(X_val_perm_scaled)

print("Validation MAE with permutation importance top 10:", mean_absolute_error(y_val, val_pred_perm))
print("Validation MSE with permutation importance top 10:", mean_squared_error(y_val, val_pred_perm))

Validation MAE with permutation importance top 10: 1045.6304869901319
Validation MSE with permutation importance top 10: 121448364.45053889


Permutation importance produced the best validation MAE among the feature selection methods tested so far. This suggests that evaluating features based on their actual impact on model performance can be more effective than relying only on coefficients or simple correlation.

At the same time, the overall differences remain relatively small, which means that the main predictive signal is already concentrated in a small core set of features.

## 6.4 SHAP

In [28]:
import sys
!{sys.executable} -m pip install shap

In [29]:
import shap

explainer = shap.Explainer(lasso, X_train_scaled)
shap_values = explainer(X_val_scaled)

shap_importance = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=feature_list
).sort_values(ascending=False)

print("SHAP feature importance ranking:")
display(shap_importance)

top_10_shap_features = shap_importance.head(10).index.tolist()

print("Top 10 features selected by SHAP:")
print(top_10_shap_features)

SHAP feature importance ranking:


bathrooms            941.436029
Doorman              563.783829
bedrooms             433.543359
DogsAllowed          171.028664
LaundryinBuilding    155.579716
LaundryinUnit        145.090154
NoFee                128.299634
HardwoodFloors       121.863984
Elevator              77.306075
CatsAllowed           62.093154
LaundryInBuilding     53.782604
HighSpeedInternet     46.572979
FitnessCenter         42.764335
Pre-War               32.491978
DiningRoom            31.851500
Terrace               27.256050
OutdoorSpace          19.924216
RoofDeck              16.612352
Dishwasher            15.070682
Balcony               11.607804
NewConstruction        4.706409
SwimmingPool           1.501151
dtype: float64

Top 10 features selected by SHAP:
['bathrooms', 'Doorman', 'bedrooms', 'DogsAllowed', 'LaundryinBuilding', 'LaundryinUnit', 'NoFee', 'HardwoodFloors', 'Elevator', 'CatsAllowed']


In [30]:
X_train_shap = train_df_fs[top_10_shap_features]
X_val_shap = val_df_fs[top_10_shap_features]

scaler_shap = StandardScaler()

X_train_shap_scaled = scaler_shap.fit_transform(X_train_shap)
X_val_shap_scaled = scaler_shap.transform(X_val_shap)

lasso_shap = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_shap.fit(X_train_shap_scaled, y_train)

val_pred_shap = lasso_shap.predict(X_val_shap_scaled)

print("Validation MAE with SHAP top 10:", mean_absolute_error(y_val, val_pred_shap))
print("Validation MSE with SHAP top 10:", mean_squared_error(y_val, val_pred_shap))

Validation MAE with SHAP top 10: 1046.7016117086241
Validation MSE with SHAP top 10: 121438056.23937577


In [31]:
import shap

explainer = shap.LinearExplainer(lasso, X_train_scaled)
shap_values = explainer.shap_values(X_val_scaled)

shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_list
).sort_values(ascending=False)

print("SHAP feature importance ranking:")
display(shap_importance)

top_10_shap_features = shap_importance.head(10).index.tolist()

print("Top 10 features selected by SHAP:")
print(top_10_shap_features)

SHAP feature importance ranking:


bathrooms            941.436029
Doorman              563.783829
bedrooms             433.543359
DogsAllowed          171.028664
LaundryinBuilding    155.579716
LaundryinUnit        145.090154
NoFee                128.299634
HardwoodFloors       121.863984
Elevator              77.306075
CatsAllowed           62.093154
LaundryInBuilding     53.782604
HighSpeedInternet     46.572979
FitnessCenter         42.764335
Pre-War               32.491978
DiningRoom            31.851500
Terrace               27.256050
OutdoorSpace          19.924216
RoofDeck              16.612352
Dishwasher            15.070682
Balcony               11.607804
NewConstruction        4.706409
SwimmingPool           1.501151
dtype: float64

Top 10 features selected by SHAP:
['bathrooms', 'Doorman', 'bedrooms', 'DogsAllowed', 'LaundryinBuilding', 'LaundryinUnit', 'NoFee', 'HardwoodFloors', 'Elevator', 'CatsAllowed']


In [32]:
feature_selection_results = pd.DataFrame({
    "method": [
        "All features",
        "Lasso top 10",
        "Nan-ratio + correlation top 10",
        "Permutation importance top 10",
        "SHAP top 10"
    ],
    "num_features": [
        len(feature_list),
        len(top_10_lasso_features),
        len(top_10_simple_features),
        len(top_10_perm_features),
        len(top_10_shap_features)
    ],
    "validation_mae": [
        mean_absolute_error(y_val, val_pred),
        mean_absolute_error(y_val, val_pred_top10),
        mean_absolute_error(y_val, val_pred_simple),
        mean_absolute_error(y_val, val_pred_perm),
        mean_absolute_error(y_val, val_pred_shap)
    ],
    "validation_mse": [
        mean_squared_error(y_val, val_pred),
        mean_squared_error(y_val, val_pred_top10),
        mean_squared_error(y_val, val_pred_simple),
        mean_squared_error(y_val, val_pred_perm),
        mean_squared_error(y_val, val_pred_shap)
    ],
    "speed_comment": [
        "baseline",
        "fast",
        "very fast",
        "medium",
        "slow"
    ],
    "stability_comment": [
        "baseline",
        "good",
        "medium",
        "good",
        "good"
    ]
})

feature_selection_results.sort_values("validation_mae")

,method,num_features,validation_mae,validation_mse,speed_comment,stability_comment
3,Permutation importance top 10,10,1045.630487,1.214484e+08,medium,good
4,SHAP top 10,10,1046.701612,1.214381e+08,slow,good
2,Nan-ratio + correlation top 10,10,1046.893507,1.215202e+08,very fast,medium
1,Lasso top 10,10,1047.641657,1.214388e+08,fast,good
0,All features,22,1048.653352,1.214108e+08,baseline,baseline


Overall, the feature selection methods produced very similar validation performance, which suggests that most of the useful signal is concentrated in a relatively small subset of features.

Among the tested methods, permutation importance produced the best validation MAE, while SHAP also performed strongly and provided a more interpretable ranking. The simple correlation-based method remained competitive, and Lasso provided a compact embedded selection baseline.

The comparison shows that all feature selection methods produced relatively similar results, which suggests that most of the predictive signal is concentrated in a small subset of features.

If I focus on validation MAE, permutation importance gave the best result while using only 10 features, which makes it the strongest compact feature selection method in this experiment.

If I focus on validation MSE, the full feature set still performed slightly better. This suggests that keeping all features may still help with some larger errors, even if the overall average absolute error is not the best.

Overall, permutation importance looks like the best practical choice here because it achieved the best MAE with a much smaller feature set, while SHAP also performed well and provided a more interpretable ranking.

# 7. Hyperparameter Optimization

In this section, I will tune the hyperparameters of an ElasticNet model. I will start with manual Grid Search and Random Search using `alpha` and `l1_ratio`, then compare the results before moving to Optuna.

In [33]:
from sklearn.linear_model import ElasticNet

X_train_hp = train_df_fs[feature_list]
X_val_hp = val_df_fs[feature_list]
X_test_hp = test_df_fs[feature_list]

y_train_hp = train_df_fs["price"]
y_val_hp = val_df_fs["price"]
y_test_hp = test_df_fs["price"]

scaler_hp = StandardScaler()

X_train_hp_scaled = scaler_hp.fit_transform(X_train_hp)
X_val_hp_scaled = scaler_hp.transform(X_val_hp)
X_test_hp_scaled = scaler_hp.transform(X_test_hp)

In [34]:
alpha_grid = [0.001, 0.01, 0.1, 1.0]
l1_ratio_grid = [0.2, 0.5, 0.8]

grid_results = []

for alpha in alpha_grid:
    for l1_ratio in l1_ratio_grid:
        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=10000,
            random_state=42
        )
        
        model.fit(X_train_hp_scaled, y_train_hp)
        val_pred_hp = model.predict(X_val_hp_scaled)
        
        mae = mean_absolute_error(y_val_hp, val_pred_hp)
        mse = mean_squared_error(y_val_hp, val_pred_hp)

        grid_results.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "validation_mae": mae,
            "validation_mse": mse
        })

grid_results_df = pd.DataFrame(grid_results).sort_values("validation_mae")
grid_results_df

,alpha,l1_ratio,validation_mae,validation_mse
10,1.000,0.5,1010.257546,1.216195e+08
11,1.000,0.8,1012.510907,1.214647e+08
6,0.100,0.2,1028.752643,1.214216e+08
9,1.000,0.2,1029.571841,1.217854e+08
7,0.100,0.5,1035.131900,1.214153e+08
8,0.100,0.8,1042.649474,1.214115e+08
3,0.010,0.2,1046.160803,1.214109e+08
4,0.010,0.5,1047.072793,1.214108e+08
5,0.010,0.8,1048.012032,1.214108e+08
0,0.001,0.2,1048.404311,1.214108e+08


In [35]:
best_grid_row = grid_results_df.iloc[0]

best_grid_alpha = best_grid_row["alpha"]
best_grid_l1_ratio = best_grid_row["l1_ratio"]

print("Best Grid Search parameters:")
print("alpha =", best_grid_alpha)
print("l1_ratio =", best_grid_l1_ratio)
print("validation_mae =", best_grid_row["validation_mae"])
print("validation_mse =", best_grid_row["validation_mse"])

Best Grid Search parameters:
alpha = 1.0
l1_ratio = 0.5
validation_mae = 1010.2575457146357
validation_mse = 121619463.34505023


In [36]:
best_grid_model = ElasticNet(
    alpha=best_grid_alpha,
    l1_ratio=best_grid_l1_ratio,
    max_iter=10000,
    random_state=42
)

best_grid_model.fit(X_train_hp_scaled, y_train_hp)

val_pred_grid = best_grid_model.predict(X_val_hp_scaled)
test_pred_grid = best_grid_model.predict(X_test_hp_scaled)

print("Best Grid Search Validation MAE:", mean_absolute_error(y_val_hp, val_pred_grid))
print("Best Grid Search Validation MSE:", mean_squared_error(y_val_hp, val_pred_grid))

print("Best Grid Search Test MAE:", mean_absolute_error(y_test_hp, test_pred_grid))
print("Best Grid Search Test MSE:", mean_squared_error(y_test_hp, test_pred_grid))

Best Grid Search Validation MAE: 1010.2575457146357
Best Grid Search Validation MSE: 121619463.34505023
Best Grid Search Test MAE: 1327.7342691994693
Best Grid Search Test MSE: 2042627284.001937


The manual Grid Search improved the validation MAE substantially, but the test MAE remained much worse than the validation result. This suggests that the selected hyperparameters worked well on the validation split, but their benefit did not generalize equally well to the final test set.

In [37]:
np.random.seed(42)

alpha_candidates = [0.001, 0.01, 0.1, 1.0]
l1_ratio_candidates = [0.2, 0.5, 0.8]

random_results = []
n_trials = 6

sampled_combinations = []

while len(sampled_combinations) < n_trials:
    combo = (
        np.random.choice(alpha_candidates),
        np.random.choice(l1_ratio_candidates)
    )
    if combo not in sampled_combinations:
        sampled_combinations.append(combo)

for alpha, l1_ratio in sampled_combinations:
    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10000,
        random_state=42
    )

    model.fit(X_train_hp_scaled, y_train_hp)
    val_pred_random = model.predict(X_val_hp_scaled)

    mae = mean_absolute_error(y_val_hp, val_pred_random)
    mse = mean_squared_error(y_val_hp, val_pred_random)

    random_results.append({
        "alpha": alpha,
        "l1_ratio": l1_ratio,
        "validation_mae": mae,
        "validation_mse": mse
    })

random_results_df = pd.DataFrame(random_results).sort_values("validation_mae")
random_results_df

,alpha,l1_ratio,validation_mae,validation_mse
5,1.000,0.8,1012.510907,1.214647e+08
0,0.100,0.2,1028.752643,1.214216e+08
2,1.000,0.2,1029.571841,1.217854e+08
1,0.100,0.8,1042.649474,1.214115e+08
4,0.010,0.8,1048.012032,1.214108e+08
3,0.001,0.8,1048.597569,1.214108e+08


In [38]:
best_random_row = random_results_df.iloc[0]

best_random_alpha = best_random_row["alpha"]
best_random_l1_ratio = best_random_row["l1_ratio"]

print("Best Random Search parameters:")
print("alpha =", best_random_alpha)
print("l1_ratio =", best_random_l1_ratio)
print("validation_mae =", best_random_row["validation_mae"])
print("validation_mse =", best_random_row["validation_mse"])

Best Random Search parameters:
alpha = 1.0
l1_ratio = 0.8
validation_mae = 1012.510906537983
validation_mse = 121464672.37982877


In [39]:
best_random_model = ElasticNet(
    alpha=best_random_alpha,
    l1_ratio=best_random_l1_ratio,
    max_iter=10000,
    random_state=42
)

best_random_model.fit(X_train_hp_scaled, y_train_hp)

val_pred_random_best = best_random_model.predict(X_val_hp_scaled)
test_pred_random_best = best_random_model.predict(X_test_hp_scaled)

print("Best Random Search Validation MAE:", mean_absolute_error(y_val_hp, val_pred_random_best))
print("Best Random Search Validation MSE:", mean_squared_error(y_val_hp, val_pred_random_best))

print("Best Random Search Test MAE:", mean_absolute_error(y_test_hp, test_pred_random_best))
print("Best Random Search Test MSE:", mean_squared_error(y_test_hp, test_pred_random_best))

Best Random Search Validation MAE: 1012.510906537983
Best Random Search Validation MSE: 121464672.37982877
Best Random Search Test MAE: 1325.567612417565
Best Random Search Test MSE: 2042303073.8418844


The comparison shows that Random Search was able to get very close to Grid Search while evaluating fewer hyperparameter combinations.

Grid Search achieved the best validation MAE, while Random Search gave a slightly better test MAE. The difference between them was small overall, which suggests that Random Search can be an efficient alternative when the search space is not too small and computational cost matters.

In [40]:
tuning_comparison_df = pd.DataFrame({
    "method": ["Grid Search", "Random Search"],
    "best_alpha": [best_grid_alpha, best_random_alpha],
    "best_l1_ratio": [best_grid_l1_ratio, best_random_l1_ratio],
    "validation_mae": [
        mean_absolute_error(y_val_hp, val_pred_grid),
        mean_absolute_error(y_val_hp, val_pred_random_best)
    ],
    "validation_mse": [
        mean_squared_error(y_val_hp, val_pred_grid),
        mean_squared_error(y_val_hp, val_pred_random_best)
    ],
    "test_mae": [
        mean_absolute_error(y_test_hp, test_pred_grid),
        mean_absolute_error(y_test_hp, test_pred_random_best)
    ],
    "test_mse": [
        mean_squared_error(y_test_hp, test_pred_grid),
        mean_squared_error(y_test_hp, test_pred_random_best)
    ]
})

tuning_comparison_df

,method,best_alpha,best_l1_ratio,validation_mae,validation_mse,test_mae,test_mse
0,Grid Search,1.0,0.5,1010.257546,1.216195e+08,1327.734269,2.042627e+09
1,Random Search,1.0,0.8,1012.510907,1.214647e+08,1325.567612,2.042303e+09


In [41]:
import sys
!{sys.executable} -m pip install optuna

In [42]:
import optuna
print(optuna.__version__)

4.8.0


In [43]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.2, 0.8)

    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10000,
        random_state=42
    )

    model.fit(X_train_hp_scaled, y_train_hp)
    val_pred = model.predict(X_val_hp_scaled)

    mae = mean_absolute_error(y_val_hp, val_pred)
    return mae


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("Best Optuna parameters:", study.best_params)
print("Best Optuna validation MAE:", study.best_value)

[I 2026-04-25 03:25:23,663] A new study created in memory with name: no-name-d3a64f25-e54b-435a-b52a-7ffa9fef26c9
[I 2026-04-25 03:25:23,693] Trial 0 finished with value: 1030.5469424097191 and parameters: {'alpha': 0.09042146888003957, 'l1_ratio': 0.2144030825468155}. Best is trial 0 with value: 1030.5469424097191.
[I 2026-04-25 03:25:23,741] Trial 1 finished with value: 1037.0987403300746 and parameters: {'alpha': 0.07767433543173782, 'l1_ratio': 0.46320004397447173}. Best is trial 0 with value: 1030.5469424097191.
[I 2026-04-25 03:25:23,792] Trial 2 finished with value: 1047.7478971636951 and parameters: {'alpha': 0.011868672846373916, 'l1_ratio': 0.7616068039002888}. Best is trial 0 with value: 1030.5469424097191.
[I 2026-04-25 03:25:23,827] Trial 3 finished with value: 1042.375146665047 and parameters: {'alpha': 0.04191015609381292, 'l1_ratio': 0.49480892304861185}. Best is trial 0 with value: 1030.5469424097191.
[I 2026-04-25 03:25:23,869] Trial 4 finished with value: 1047.097027

Best Optuna parameters: {'alpha': 0.42618147387876476, 'l1_ratio': 0.42246238237950107}
Best Optuna validation MAE: 1009.6490044287035


In [44]:
best_optuna_alpha = study.best_params["alpha"]
best_optuna_l1_ratio = study.best_params["l1_ratio"]

best_optuna_model = ElasticNet(
    alpha=best_optuna_alpha,
    l1_ratio=best_optuna_l1_ratio,
    max_iter=10000,
    random_state=42
)

best_optuna_model.fit(X_train_hp_scaled, y_train_hp)

val_pred_optuna = best_optuna_model.predict(X_val_hp_scaled)
test_pred_optuna = best_optuna_model.predict(X_test_hp_scaled)

print("Best Optuna Validation MAE:", mean_absolute_error(y_val_hp, val_pred_optuna))
print("Best Optuna Validation MSE:", mean_squared_error(y_val_hp, val_pred_optuna))

print("Best Optuna Test MAE:", mean_absolute_error(y_test_hp, test_pred_optuna))
print("Best Optuna Test MSE:", mean_squared_error(y_test_hp, test_pred_optuna))

Best Optuna Validation MAE: 1009.6490044287035
Best Optuna Validation MSE: 121484976.88268736
Best Optuna Test MAE: 1323.563919975567
Best Optuna Test MSE: 2042353123.6585822


Optuna produced the best validation MAE among all hyperparameter search methods. It also achieved the best test MAE, which suggests that its selected configuration generalized slightly better than the configurations found by Grid Search and Random Search.

This result is consistent with the idea behind Bayesian optimization: instead of checking combinations blindly, it uses previous trials to focus the search on more promising regions of the hyperparameter space.

In [45]:
final_tuning_comparison_df = pd.DataFrame({
    "method": ["Grid Search", "Random Search", "Optuna"],
    "best_alpha": [
        best_grid_alpha,
        best_random_alpha,
        best_optuna_alpha
    ],
    "best_l1_ratio": [
        best_grid_l1_ratio,
        best_random_l1_ratio,
        best_optuna_l1_ratio
    ],
    "validation_mae": [
        mean_absolute_error(y_val_hp, val_pred_grid),
        mean_absolute_error(y_val_hp, val_pred_random_best),
        mean_absolute_error(y_val_hp, val_pred_optuna)
    ],
    "validation_mse": [
        mean_squared_error(y_val_hp, val_pred_grid),
        mean_squared_error(y_val_hp, val_pred_random_best),
        mean_squared_error(y_val_hp, val_pred_optuna)
    ],
    "test_mae": [
        mean_absolute_error(y_test_hp, test_pred_grid),
        mean_absolute_error(y_test_hp, test_pred_random_best),
        mean_absolute_error(y_test_hp, test_pred_optuna)
    ],
    "test_mse": [
        mean_squared_error(y_test_hp, test_pred_grid),
        mean_squared_error(y_test_hp, test_pred_random_best),
        mean_squared_error(y_test_hp, test_pred_optuna)
    ]
})

final_tuning_comparison_df.sort_values("test_mae")

,method,best_alpha,best_l1_ratio,validation_mae,validation_mse,test_mae,test_mse
2,Optuna,0.426181,0.422462,1009.649004,1.214850e+08,1323.563920,2.042353e+09
1,Random Search,1.000000,0.800000,1012.510907,1.214647e+08,1325.567612,2.042303e+09
0,Grid Search,1.000000,0.500000,1010.257546,1.216195e+08,1327.734269,2.042627e+09


Among the three tuning strategies, Optuna gave the best overall result in this experiment. Grid Search remained a strong baseline, while Random Search achieved competitive performance with fewer evaluated combinations.

Overall, the tuning results suggest that a smarter search strategy can provide a small but meaningful improvement over fixed-grid and random exploration.

## 7.4 Optuna with Cross-Validation

To make hyperparameter tuning more robust, I will also run Optuna using a cross-validation scheme instead of a single validation split. In this case, I will use K-Fold and optimize the average validation MAE across folds.

In [46]:
from sklearn.model_selection import KFold

def objective_cv(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.2, 0.8)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_maes = []

    for train_idx, val_idx in kf.split(model_df):
        train_fold = model_df.iloc[train_idx]
        val_fold = model_df.iloc[val_idx]

        X_train_fold = train_fold[feature_list]
        y_train_fold = train_fold["price"]

        X_val_fold = val_fold[feature_list]
        y_val_fold = val_fold["price"]

        scaler_fold = StandardScaler()

        X_train_fold_scaled = scaler_fold.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler_fold.transform(X_val_fold)

        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=10000,
            random_state=42
        )

        model.fit(X_train_fold_scaled, y_train_fold)
        val_pred_fold = model.predict(X_val_fold_scaled)

        fold_mae = mean_absolute_error(y_val_fold, val_pred_fold)
        fold_maes.append(fold_mae)

    return np.mean(fold_maes)


study_cv = optuna.create_study(direction="minimize")
study_cv.optimize(objective_cv, n_trials=20)

print("Best Optuna CV parameters:", study_cv.best_params)
print("Best Optuna CV mean MAE:", study_cv.best_value)

[I 2026-04-25 03:25:24,457] A new study created in memory with name: no-name-7305fe02-7f95-43c5-b067-364ca587a76e
[I 2026-04-25 03:25:24,968] Trial 0 finished with value: 1161.570420327881 and parameters: {'alpha': 0.001586232386663541, 'l1_ratio': 0.6680164689602821}. Best is trial 0 with value: 1161.570420327881.
[I 2026-04-25 03:25:25,457] Trial 1 finished with value: 1161.275741257376 and parameters: {'alpha': 0.0013865057077690093, 'l1_ratio': 0.24388632864770193}. Best is trial 1 with value: 1161.275741257376.
[I 2026-04-25 03:25:25,758] Trial 2 finished with value: 1147.6032306158118 and parameters: {'alpha': 0.12542972260028393, 'l1_ratio': 0.7810485494841839}. Best is trial 2 with value: 1147.6032306158118.
[I 2026-04-25 03:25:25,961] Trial 3 finished with value: 1103.4067185742615 and parameters: {'alpha': 0.5764447233396239, 'l1_ratio': 0.7174499846433147}. Best is trial 3 with value: 1103.4067185742615.
[I 2026-04-25 03:25:26,376] Trial 4 finished with value: 1160.753814392

Best Optuna CV parameters: {'alpha': 0.9645958027632527, 'l1_ratio': 0.5859749333916282}
Best Optuna CV mean MAE: 1077.9211197135523


In [47]:
best_optuna_cv_alpha = study_cv.best_params["alpha"]
best_optuna_cv_l1_ratio = study_cv.best_params["l1_ratio"]

best_optuna_cv_model = ElasticNet(
    alpha=best_optuna_cv_alpha,
    l1_ratio=best_optuna_cv_l1_ratio,
    max_iter=10000,
    random_state=42
)

best_optuna_cv_model.fit(X_train_hp_scaled, y_train_hp)

val_pred_optuna_cv = best_optuna_cv_model.predict(X_val_hp_scaled)
test_pred_optuna_cv = best_optuna_cv_model.predict(X_test_hp_scaled)

print("Optuna CV Validation MAE:", mean_absolute_error(y_val_hp, val_pred_optuna_cv))
print("Optuna CV Validation MSE:", mean_squared_error(y_val_hp, val_pred_optuna_cv))

print("Optuna CV Test MAE:", mean_absolute_error(y_test_hp, test_pred_optuna_cv))
print("Optuna CV Test MSE:", mean_squared_error(y_test_hp, test_pred_optuna_cv))

Optuna CV Validation MAE: 1007.3812131451024
Optuna CV Validation MSE: 121564086.51466407
Optuna CV Test MAE: 1323.624172370419
Optuna CV Test MSE: 2042523687.5616107


In [48]:
extended_tuning_comparison_df = pd.DataFrame({
    "method": ["Grid Search", "Random Search", "Optuna", "Optuna + KFold CV"],
    "best_alpha": [
        best_grid_alpha,
        best_random_alpha,
        best_optuna_alpha,
        best_optuna_cv_alpha
    ],
    "best_l1_ratio": [
        best_grid_l1_ratio,
        best_random_l1_ratio,
        best_optuna_l1_ratio,
        best_optuna_cv_l1_ratio
    ],
    "validation_mae": [
        mean_absolute_error(y_val_hp, val_pred_grid),
        mean_absolute_error(y_val_hp, val_pred_random_best),
        mean_absolute_error(y_val_hp, val_pred_optuna),
        mean_absolute_error(y_val_hp, val_pred_optuna_cv)
    ],
    "validation_mse": [
        mean_squared_error(y_val_hp, val_pred_grid),
        mean_squared_error(y_val_hp, val_pred_random_best),
        mean_squared_error(y_val_hp, val_pred_optuna),
        mean_squared_error(y_val_hp, val_pred_optuna_cv)
    ],
    "test_mae": [
        mean_absolute_error(y_test_hp, test_pred_grid),
        mean_absolute_error(y_test_hp, test_pred_random_best),
        mean_absolute_error(y_test_hp, test_pred_optuna),
        mean_absolute_error(y_test_hp, test_pred_optuna_cv)
    ],
    "test_mse": [
        mean_squared_error(y_test_hp, test_pred_grid),
        mean_squared_error(y_test_hp, test_pred_random_best),
        mean_squared_error(y_test_hp, test_pred_optuna),
        mean_squared_error(y_test_hp, test_pred_optuna_cv)
    ]
})

extended_tuning_comparison_df.sort_values("test_mae")

,method,best_alpha,best_l1_ratio,validation_mae,validation_mse,test_mae,test_mse
2,Optuna,0.426181,0.422462,1009.649004,1.214850e+08,1323.563920,2.042353e+09
3,Optuna + KFold CV,0.964596,0.585975,1007.381213,1.215641e+08,1323.624172,2.042524e+09
1,Random Search,1.000000,0.800000,1012.510907,1.214647e+08,1325.567612,2.042303e+09
0,Grid Search,1.000000,0.500000,1010.257546,1.216195e+08,1327.734269,2.042627e+09


Using Optuna with cross-validation makes the hyperparameter search more robust because the objective is no longer based on a single split. Instead, the selected parameters are optimized based on average performance across multiple folds.

This makes the tuning process more reliable and better aligned with the overall goal of the project, which is to build a fair evaluation pipeline rather than optimize for one convenient split.

One important detail here is that the Optuna + KFold CV objective is not directly comparable to the single-split validation objective used in the earlier tuning experiments.

In the first case, the hyperparameters were selected based on one validation split. In the cross-validation version, they were selected based on the average MAE across multiple folds, which makes the tuning process more robust but also changes the meaning of the optimization score.

In this experiment, the standard Optuna run gave the best holdout validation and test MAE, while Optuna with K-Fold provided a more conservative and methodologically stronger tuning setup.

# 8. Final Comparison and Conclusion

At this stage, I can summarize the main findings from the project and compare the different choices I made for validation, feature selection, and hyperparameter tuning. The goal is to identify the most reasonable final pipeline rather than focus on one metric in isolation.

In [49]:
final_summary_df = pd.DataFrame({
    "component": [
        "Validation scheme",
        "Feature selection",
        "Hyperparameter tuning"
    ],
    "selected_choice": [
        "Time-aware validation / TimeSeries-style logic",
        "Permutation importance top 10",
        "Optuna"
    ],
    "reason": [
        "Most realistic when time ordering matters and helps avoid future leakage.",
        "Best validation MAE among compact feature subsets with only 10 features.",
        "Best holdout validation and test MAE among the tuning methods."
    ]
})

final_summary_df

,component,selected_choice,reason
0,Validation scheme,Time-aware validation / TimeSeries-style logic,Most realistic when time ordering matters and ...
1,Feature selection,Permutation importance top 10,Best validation MAE among compact feature subs...
2,Hyperparameter tuning,Optuna,Best holdout validation and test MAE among the...


## Final conclusion

This project was not only about training a model, but about designing a reliable evaluation pipeline.

The main lesson for me is that model quality depends heavily on how the evaluation process is designed. A good score is not enough by itself. It only becomes meaningful when the split logic is appropriate, leakage is avoided, feature selection is done fairly, and hyperparameters are tuned in a controlled way.

From the validation experiments, time-aware splitting appears to be the most realistic choice when temporal order matters. From the feature selection experiments, permutation importance gave the best compact subset in terms of validation MAE. From the tuning experiments, Optuna produced the best overall result on the holdout validation and test sets.

Overall, the final pipeline I would keep from this notebook is:
- time-aware validation when realism matters,
- permutation importance for compact feature selection,
- and Optuna for hyperparameter optimization.

The broader conclusion is that careful evaluation design is just as important as the model itself, and in many practical cases, it matters even more than trying a more complex algorithm.